# NLQuery - Natural Language Table Question Answering
## Section 1: Setup, Data Pipeline & Drive Caching

**Team:** Vrushabh, Jeni Savalia, Mahak Gupta

**Project Overview:**
- Dataset: WikiTableQuestions (22K QA pairs, 2,108 Wikipedia tables)
- Task: Natural Language -> SQL -> Execute -> Answer
- Approaches: Prompt Engineering, RAG, Fine-Tuning

---

### How This Notebook Works
- **First run:** Cells download data, process tables, build SQLite DBs, and cache everything to Google Drive.
- **Subsequent runs:** Cells detect cached files on Drive and skip expensive processing.
- **Rule:** Always run cells in order from top to bottom on a fresh session.

---
## 1.1 Install Dependencies

In [1]:
%%capture
!pip install datasets==2.14.7 transformers==4.46.0 huggingface_hub==0.26.0 accelerate "bitsandbytes>=0.46.1" sentence-transformers faiss-gpu peft trl gradio sqlparse
print("All dependencies installed!")

---
## 1.2 Imports

In [2]:
import os
import json
import sqlite3
import pickle
import time
import re
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from tqdm.notebook import tqdm

from datasets import load_dataset

print("All imports loaded successfully!")

All imports loaded successfully!


---
## 1.3 Mount Google Drive & Create Folder Structure

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/NLQuery'

subfolders = [
    'data',
    'models',
    'embeddings',
    'results/baseline_template',
    'results/baseline_zeroshot',
    'results/fewshot',
    'results/cot',
    'results/rag',
    'results/finetuned',
    'plots'
]

for folder in subfolders:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)

print(f"Project folder structure created at: {BASE_PATH}")
print()
print("Folder structure:")
for folder in subfolders:
    print(f"   {folder}/")

Mounted at /content/drive
Project folder structure created at: /content/drive/MyDrive/NLQuery

Folder structure:
   data/
   models/
   embeddings/
   results/baseline_template/
   results/baseline_zeroshot/
   results/fewshot/
   results/cot/
   results/rag/
   results/finetuned/
   plots/


---
## 1.4 Helper: Cache Manager
Saves and loads expensive outputs to/from Drive so you don't re-run them.

In [4]:
class CacheManager:
    def __init__(self, base_path):
        self.base_path = base_path

    def cache_path(self, subfolder, filename):
        return os.path.join(self.base_path, subfolder, filename)

    def exists(self, subfolder, filename):
        return os.path.exists(self.cache_path(subfolder, filename))

    def save_pickle(self, data, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        with open(path, 'wb') as f:
            pickle.dump(data, f)
        print(f"Saved to Drive: {subfolder}/{filename}")

    def load_pickle(self, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        with open(path, 'rb') as f:
            data = pickle.load(f)
        print(f"Loaded from Drive: {subfolder}/{filename}")
        return data

    def save_json(self, data, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"Saved to Drive: {subfolder}/{filename}")

    def load_json(self, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        with open(path, 'r') as f:
            data = json.load(f)
        print(f"Loaded from Drive: {subfolder}/{filename}")
        return data

    def save_csv(self, df, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        df.to_csv(path, index=False)
        print(f"Saved to Drive: {subfolder}/{filename}")

    def load_csv(self, subfolder, filename):
        path = self.cache_path(subfolder, filename)
        df = pd.read_csv(path)
        print(f"Loaded from Drive: {subfolder}/{filename}")
        return df

cache = CacheManager(BASE_PATH)
print("Cache manager ready!")

Cache manager ready!


---
## 1.5 Load WikiTableQuestions Dataset

In [5]:
print("Loading WikiTableQuestions from Stanford NLP...")

!wget -q https://github.com/ppasupat/WikiTableQuestions/releases/download/v1.0.2/WikiTableQuestions-1.0.2-compact.zip -O /tmp/wtq.zip
!unzip -qo /tmp/wtq.zip -d /tmp/wtq

import csv

BASE_DIR = '/tmp/wtq/WikiTableQuestions'

def load_wtq_from_files():
    data = {'train': [], 'test': []}

    splits = {
        'train': f'{BASE_DIR}/data/random-split-1-train.tsv',
        'test': f'{BASE_DIR}/data/random-split-1-dev.tsv'
    }

    for split, filepath in splits.items():
        with open(filepath, 'r') as f:
            reader = csv.DictReader(f, delimiter='\t')
            for row in reader:
                table_path = f'{BASE_DIR}/{row["context"]}'
                header = []
                rows = []
                try:
                    with open(table_path, 'r', encoding='utf-8', errors='replace') as tf:
                        # Detect delimiter based on file extension
                        if table_path.endswith('.csv'):
                            table_reader = csv.reader(tf, delimiter=',')
                        else:
                            table_reader = csv.reader(tf, delimiter='\t')
                        header = next(table_reader)
                        rows = [r for r in table_reader]
                except:
                    continue

                # Skip tables with bad parsing
                if len(header) <= 1:
                    continue

                answer = row['targetValue']
                if '|' in answer:
                    answers = [a.strip() for a in answer.split('|')]
                else:
                    answers = [answer]

                data[split].append({
                    'id': row['id'],
                    'question': row['utterance'],
                    'answers': answers,
                    'table': {
                        'header': header,
                        'rows': rows,
                        'name': row.get('context', '')
                    }
                })
    return data

wtq_data = load_wtq_from_files()

from datasets import Dataset, DatasetDict
raw_dataset = DatasetDict({
    'train': Dataset.from_list(wtq_data['train']),
    'test': Dataset.from_list(wtq_data['test'])
})

print()
print("Dataset loaded!")
print(f"   Train: {len(raw_dataset['train'])} examples")
print(f"   Test:  {len(raw_dataset['test'])} examples")
print(f"   Total: {len(raw_dataset['train']) + len(raw_dataset['test'])} examples")

# Verify table parsing is correct
sample = raw_dataset['train'][0]
print(f"\nVerification:")
print(f"   Question: {sample['question']}")
print(f"   Answers: {sample['answers']}")
print(f"   Table columns: {sample['table']['header']}")
print(f"   First row: {sample['table']['rows'][0]}")
print(f"   Num columns: {len(sample['table']['header'])}")

Loading WikiTableQuestions from Stanford NLP...

Dataset loaded!
   Train: 11321 examples
   Test:  2810 examples
   Total: 14131 examples

Verification:
   Question: what was the last year where this team was a part of the usl a-league?
   Answers: ['2004']
   Table columns: ['Year', 'Division', 'League', 'Regular Season', 'Playoffs', 'Open Cup', 'Avg. Attendance']
   First row: ['2001', '2', 'USL A-League', '4th, Western', 'Quarterfinals', 'Did not qualify', '7,169']
   Num columns: 7


---
## 1.6 Process Tables to Pandas DataFrames

Key steps:
1. Convert each table to a Pandas DataFrame
2. Clean column names for SQL compatibility
3. Infer data types (numeric vs string)
4. Create SQLite databases for SQL execution

In [6]:
def clean_column_name(col_name):
    cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', col_name.strip())
    cleaned = cleaned.strip('_')
    cleaned = re.sub(r'_+', '_', cleaned)
    if cleaned and cleaned[0].isdigit():
        cleaned = 'col_' + cleaned
    if not cleaned:
        cleaned = 'unnamed_col'
    return cleaned.lower()


def try_convert_numeric(value):
    if pd.isna(value) or value == '' or value is None:
        return None
    cleaned = str(value).replace(',', '').replace('$', '').replace('%', '').strip()
    try:
        if '.' not in cleaned:
            return int(cleaned)
        return float(cleaned)
    except (ValueError, TypeError):
        return value


def table_to_dataframe(table_data):
    header = table_data['header']
    rows = table_data['rows']

    clean_cols = [clean_column_name(col) for col in header]

    # Handle duplicate column names
    seen = {}
    unique_cols = []
    for col in clean_cols:
        if col in seen:
            seen[col] += 1
            unique_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            unique_cols.append(col)

    num_cols = len(unique_cols)

    # Fix rows that don't match header length
    fixed_rows = []
    for row in rows:
        if len(row) == num_cols:
            fixed_rows.append(row)
        elif len(row) > num_cols:
            fixed_rows.append(row[:num_cols])
        elif len(row) < num_cols:
            fixed_rows.append(row + [''] * (num_cols - len(row)))

    df = pd.DataFrame(fixed_rows, columns=unique_cols)

    # Try to convert numeric columns
    for col in df.columns:
        converted = df[col].apply(try_convert_numeric)
        numeric_count = sum(1 for v in converted if isinstance(v, (int, float)) and v is not None)
        non_null_count = sum(1 for v in converted if v is not None)
        if non_null_count > 0 and numeric_count / non_null_count > 0.5:
            df[col] = pd.to_numeric(converted, errors='coerce')

    return df, header, unique_cols


print("Table processing functions defined!")

Table processing functions defined!


In [7]:
sample = raw_dataset['train'][0]
df_test, orig_cols, clean_cols = table_to_dataframe(sample['table'])

print("Original columns:", orig_cols)
print("Cleaned columns: ", clean_cols)
print(f"DataFrame shape: {df_test.shape}")
print()
print("Data types:")
print(df_test.dtypes)
print()
print("Preview:")
display(df_test.head())

Original columns: ['Year', 'Division', 'League', 'Regular Season', 'Playoffs', 'Open Cup', 'Avg. Attendance']
Cleaned columns:  ['year', 'division', 'league', 'regular_season', 'playoffs', 'open_cup', 'avg_attendance']
DataFrame shape: (10, 7)

Data types:
year               int64
division           int64
league            object
regular_season    object
playoffs          object
open_cup          object
avg_attendance     int64
dtype: object

Preview:


,year,division,league,regular_season,playoffs,open_cup,avg_attendance
0,2001,2,USL A-League,"4th, Western",Quarterfinals,Did not qualify,7169
1,2002,2,USL A-League,"2nd, Pacific",1st Round,Did not qualify,6260
2,2003,2,USL A-League,"3rd, Pacific",Did not qualify,Did not qualify,5871
3,2004,2,USL A-League,"1st, Western",Quarterfinals,4th Round,5628
4,2005,2,USL First Division,5th,Quarterfinals,4th Round,6028


---
## 1.7 Build Processed Dataset with Table Metadata

For each question-answer pair, we store:
- The question and gold answer(s)
- The table as a DataFrame
- Original and cleaned column names
- Table schema info (column types, sample values)
- A unique table ID for SQLite lookup

In [8]:
def process_split(dataset_split, split_name):
    processed_examples = []
    tables = {}

    for idx in tqdm(range(len(dataset_split)), desc=f"Processing {split_name}"):
        item = dataset_split[idx]
        question = item['question']
        answers = item['answers']
        table_data = item['table']

        table_id = f"{split_name}_{idx}"
        table_id = re.sub(r'[^a-zA-Z0-9_]', '_', table_id)

        if table_id not in tables:
            df, orig_cols, clean_cols = table_to_dataframe(table_data)

            schema = []
            for orig, clean in zip(orig_cols, clean_cols):
                col_type = str(df[clean].dtype) if clean in df.columns else 'unknown'
                sample_vals = df[clean].dropna().head(3).tolist() if clean in df.columns else []
                schema.append({
                    'original_name': orig,
                    'clean_name': clean,
                    'dtype': col_type,
                    'sample_values': [str(v) for v in sample_vals]
                })

            tables[table_id] = {
                'df': df,
                'orig_cols': orig_cols,
                'clean_cols': clean_cols,
                'schema': schema,
                'num_rows': len(df),
                'num_cols': len(clean_cols)
            }

        processed_examples.append({
            'id': f"{split_name}_{idx}",
            'question': question,
            'answers': answers,
            'table_id': table_id,
            'split': split_name
        })

    return processed_examples, tables


print("Processing function defined!")

Processing function defined!


In [9]:
if cache.exists('data', 'processed_train.pkl') and cache.exists('data', 'processed_test.pkl'):
    print("Loading processed data from Drive cache...")
    train_data = cache.load_pickle('data', 'processed_train.pkl')
    test_data = cache.load_pickle('data', 'processed_test.pkl')
    train_examples = train_data['examples']
    train_tables = train_data['tables']
    test_examples = test_data['examples']
    test_tables = test_data['tables']
else:
    print("Processing dataset (first time only)...")
    train_examples, train_tables = process_split(raw_dataset['train'], 'train')
    test_examples, test_tables = process_split(raw_dataset['test'], 'test')

    cache.save_pickle({'examples': train_examples, 'tables': train_tables}, 'data', 'processed_train.pkl')
    cache.save_pickle({'examples': test_examples, 'tables': test_tables}, 'data', 'processed_test.pkl')

print()
print("Processing complete!")
print(f"   Train: {len(train_examples)} examples, {len(train_tables)} unique tables")
print(f"   Test:  {len(test_examples)} examples, {len(test_tables)} unique tables")

Loading processed data from Drive cache...
Loaded from Drive: data/processed_train.pkl
Loaded from Drive: data/processed_test.pkl

Processing complete!
   Train: 11321 examples, 11321 unique tables
   Test:  2810 examples, 2810 unique tables


---
## 1.8 Create SQLite Databases

Each table gets stored in a SQLite database so we can execute generated SQL queries against it.

In [10]:
def create_sqlite_db(tables_dict, db_path):
    conn = sqlite3.connect(db_path)
    success_count = 0
    error_count = 0

    for table_id, table_info in tqdm(tables_dict.items(), desc="Creating SQLite tables"):
        try:
            df = table_info['df']
            df.to_sql(table_id, conn, if_exists='replace', index=False)
            success_count += 1
        except Exception as e:
            error_count += 1
            if error_count <= 5:
                print(f"   Warning: {table_id}: {str(e)[:100]}")

    conn.close()
    print(f"SQLite DB created: {success_count} tables loaded, {error_count} errors")
    return success_count, error_count


print("SQLite creation function defined!")

SQLite creation function defined!


In [11]:
TRAIN_DB_PATH = cache.cache_path('data', 'train_tables.db')
TEST_DB_PATH = cache.cache_path('data', 'test_tables.db')

if os.path.exists(TRAIN_DB_PATH) and os.path.exists(TEST_DB_PATH):
    print("SQLite databases already exist on Drive. Skipping creation.")
else:
    print("Creating SQLite databases (first time only)...")
    print("--- Train DB ---")
    create_sqlite_db(train_tables, TRAIN_DB_PATH)
    print("--- Test DB ---")
    create_sqlite_db(test_tables, TEST_DB_PATH)

print()
print(f"Train DB: {TRAIN_DB_PATH}")
print(f"Test DB:  {TEST_DB_PATH}")

SQLite databases already exist on Drive. Skipping creation.

Train DB: /content/drive/MyDrive/NLQuery/data/train_tables.db
Test DB:  /content/drive/MyDrive/NLQuery/data/test_tables.db


---
## 1.9 SQL Execution Engine

Core function used across ALL approaches - takes generated SQL, runs it, returns result.

In [12]:
def execute_sql(sql_query, table_id, db_path, timeout=10):
    try:
        conn = sqlite3.connect(db_path)
        conn.execute("PRAGMA busy_timeout = " + str(timeout * 1000))

        modified_sql = sql_query
        for placeholder in ['FROM table', 'FROM w', 'FROM t ', 'JOIN table', 'JOIN w', 'JOIN t ']:
            if placeholder.lower() in modified_sql.lower():
                replacement = placeholder.split()[0] + ' "' + table_id + '"'
                modified_sql = re.sub(
                    re.escape(placeholder),
                    replacement,
                    modified_sql,
                    flags=re.IGNORECASE
                )

        cursor = conn.execute(modified_sql)
        rows = cursor.fetchall()
        conn.close()

        if not rows:
            return [], True, None

        if len(rows[0]) == 1:
            result = [str(row[0]) for row in rows]
        else:
            result = [str(val) for row in rows for val in row]

        return result, True, None

    except Exception as e:
        return [], False, str(e)


print("SQL execution engine ready!")

SQL execution engine ready!


In [13]:
test_example = test_examples[0]
test_table_id = test_example['table_id']
test_table_info = test_tables[test_table_id]

print(f"Question: {test_example['question']}")
print(f"Gold Answer: {test_example['answers']}")
print(f"Table ID: {test_table_id}")
print(f"Columns: {test_table_info['clean_cols']}")
print(f"Rows: {test_table_info['num_rows']}")
print()
print("Table preview:")
display(test_table_info['df'].head())

simple_sql = 'SELECT * FROM "' + test_table_id + '" LIMIT 3'
result, success, error = execute_sql(simple_sql, test_table_id, TEST_DB_PATH)
print()
print(f"Test query: {simple_sql}")
print(f"   Success: {success}")
print(f"   Result: {result[:10]}")

Question: which team won previous to crettyard?
Gold Answer: ['Wolfe Tones']
Table ID: test_0
Columns: ['team', 'county', 'wins', 'years_won']
Rows: 9

Table preview:


,team,county,wins,years_won
0,Greystones,Wicklow,1,2011
1,Ballymore Eustace,Kildare,1,2010
2,Maynooth,Kildare,1,2009
3,Ballyroan Abbey,Laois,1,2008
4,Fingal Ravens,Dublin,1,2007



Test query: SELECT * FROM "test_0" LIMIT 3
   Success: True
   Result: ['Greystones', 'Wicklow', '1', '2011', 'Ballymore Eustace', 'Kildare', '1', '2010', 'Maynooth', 'Kildare']


---
## 1.10 Evaluation Framework

Core metric: **Execution Accuracy** - does the SQL result match the gold answer?

In [14]:
def normalize_answer(answer):
    if answer is None:
        return ''
    answer = str(answer).strip().lower()
    if answer.endswith('.0'):
        answer = answer[:-2]
    answer = answer.replace(',', '')
    return answer


def check_answer(predicted_results, gold_answers):
    if not predicted_results:
        return False, 'empty'

    pred_normalized = [normalize_answer(p) for p in predicted_results]
    gold_normalized = [normalize_answer(g) for g in gold_answers]

    if sorted(pred_normalized) == sorted(gold_normalized):
        return True, 'exact'

    for pred in pred_normalized:
        if pred in gold_normalized:
            return True, 'subset'

    if len(pred_normalized) >= 1 and len(gold_normalized) == 1:
        if pred_normalized[0] == gold_normalized[0]:
            return True, 'exact'

    return False, 'none'


def evaluate_predictions(predictions, examples, db_path, tables_dict):
    example_lookup = {ex['id']: ex for ex in examples}

    total = len(predictions)
    correct = 0
    sql_valid = 0
    errors = []
    details = []

    for pred in tqdm(predictions, desc="Evaluating"):
        example = example_lookup[pred['id']]
        table_id = example['table_id']
        gold_answers = example['answers']

        result, success, error = execute_sql(pred['sql'], table_id, db_path)

        if success:
            sql_valid += 1
            match, match_type = check_answer(result, gold_answers)
            if match:
                correct += 1
        else:
            match = False
            match_type = 'sql_error'
            errors.append({'id': pred['id'], 'error': error, 'sql': pred['sql']})

        details.append({
            'id': pred['id'],
            'question': example['question'],
            'gold_answers': gold_answers,
            'predicted_sql': pred['sql'],
            'predicted_result': result if success else None,
            'sql_valid': success,
            'correct': match,
            'match_type': match_type,
            'error': error,
            'latency': pred.get('latency', None)
        })

    has_latency = any(d['latency'] is not None for d in details)
    avg_lat = float(np.mean([d['latency'] for d in details if d['latency'] is not None])) if has_latency else 0.0

    metrics = {
        'total': total,
        'execution_accuracy': correct / total if total > 0 else 0,
        'sql_validity_rate': sql_valid / total if total > 0 else 0,
        'correct_count': correct,
        'sql_valid_count': sql_valid,
        'avg_latency': avg_lat,
        'error_count': len(errors)
    }

    return metrics, details


print("Evaluation framework ready!")

Evaluation framework ready!


---
## 1.11 Query Complexity Classification

Classify each question into Simple / Medium / Complex based on linguistic patterns.

In [15]:
def classify_complexity(question, answers, table_info):
    q = question.lower()
    score = 0

    medium_keywords = ['how many', 'total', 'count', 'average', 'sum',
                       'more than', 'less than', 'between', 'before', 'after',
                       'greater', 'at least', 'maximum', 'minimum']

    complex_keywords = ['difference', 'compared to', 'both', 'and also',
                        'highest', 'lowest', 'most', 'least', 'rank',
                        'same as', 'different from', 'top', 'bottom',
                        'first and last', 'how many more', 'how many fewer',
                        'percentage', 'ratio', 'consecutive']

    for kw in medium_keywords:
        if kw in q:
            score += 1

    for kw in complex_keywords:
        if kw in q:
            score += 2

    if len(answers) > 1:
        score += 1
    if len(question.split()) > 15:
        score += 1
    if table_info['num_rows'] > 20:
        score += 1

    if score <= 1:
        return 'simple'
    elif score <= 3:
        return 'medium'
    else:
        return 'complex'


print("Complexity classifier defined!")

Complexity classifier defined!


In [16]:
if cache.exists('data', 'complexity_labels.json'):
    complexity_labels = cache.load_json('data', 'complexity_labels.json')
else:
    print("Classifying question complexity...")
    complexity_labels = {}

    for ex in tqdm(train_examples + test_examples, desc="Classifying"):
        split_name = ex['split']
        tables_ref = train_tables if split_name == 'train' else test_tables
        table_info = tables_ref[ex['table_id']]
        complexity = classify_complexity(ex['question'], ex['answers'], table_info)
        complexity_labels[ex['id']] = complexity

    cache.save_json(complexity_labels, 'data', 'complexity_labels.json')

complexity_counts = Counter(complexity_labels.values())
print()
print("Complexity Distribution:")
for level in ['simple', 'medium', 'complex']:
    count = complexity_counts.get(level, 0)
    pct = count / len(complexity_labels) * 100
    print(f"   {level.capitalize():10s}: {count:5d} ({pct:.1f}%)")

Loaded from Drive: data/complexity_labels.json

Complexity Distribution:
   Simple    :  7861 (55.6%)
   Medium    :  5117 (36.2%)
   Complex   :  1153 (8.2%)


---
## 1.12 Create Evaluation Subset

Stratified subset of 500 test examples for quick iteration.

In [17]:
def create_stratified_subset(examples, complexity_labels, n=500, seed=42):
    np.random.seed(seed)

    groups = defaultdict(list)
    for ex in examples:
        complexity = complexity_labels.get(ex['id'], 'simple')
        groups[complexity].append(ex)

    total = len(examples)
    subset = []
    for level, group_examples in groups.items():
        k = max(1, int(n * len(group_examples) / total))
        k = min(k, len(group_examples))
        indices = np.random.choice(len(group_examples), k, replace=False)
        subset.extend([group_examples[i] for i in indices])

    np.random.shuffle(subset)
    return subset[:n]


if cache.exists('data', 'eval_subset_500.json'):
    eval_subset = cache.load_json('data', 'eval_subset_500.json')
else:
    eval_subset = create_stratified_subset(test_examples, complexity_labels, n=500)
    cache.save_json(eval_subset, 'data', 'eval_subset_500.json')

subset_complexity = Counter(complexity_labels.get(ex['id'], 'simple') for ex in eval_subset)
print(f"Quick Eval Subset ({len(eval_subset)} examples):")
for level in ['simple', 'medium', 'complex']:
    count = subset_complexity.get(level, 0)
    print(f"   {level.capitalize():10s}: {count}")

Loaded from Drive: data/eval_subset_500.json
Quick Eval Subset (499 examples):
   Simple    : 279
   Medium    : 180
   Complex   : 40


---
## 1.13 Schema Formatter (Used by All Approaches)

Creates a text representation of a table schema for LLM prompts.

In [18]:
def format_schema(table_info, include_samples=True, max_sample_rows=3):
    lines = []
    lines.append("Table has " + str(table_info['num_rows']) + " rows and " + str(table_info['num_cols']) + " columns.")
    lines.append("Columns:")

    for col_info in table_info['schema']:
        dtype = 'text' if 'object' in col_info['dtype'] else 'numeric'
        samples = ', '.join(col_info['sample_values'][:3])
        lines.append("  - " + col_info['clean_name'] + " (" + dtype + ") | Examples: " + samples)

    if include_samples:
        lines.append("")
        lines.append("Sample rows (first " + str(max_sample_rows) + "):")
        df = table_info['df']
        sample_df = df.head(max_sample_rows)
        lines.append(sample_df.to_string(index=False))

    return '\n'.join(lines)


sample_table_id = test_examples[0]['table_id']
sample_table_info = test_tables[sample_table_id]
print("Schema format example:")
print("=" * 60)
print(format_schema(sample_table_info))
print("=" * 60)

Schema format example:
Table has 9 rows and 4 columns.
Columns:
  - team (text) | Examples: Greystones, Ballymore Eustace, Maynooth
  - county (text) | Examples: Wicklow, Kildare, Kildare
  - wins (numeric) | Examples: 1, 1, 1
  - years_won (numeric) | Examples: 2011, 2010, 2009

Sample rows (first 3):
             team  county  wins  years_won
       Greystones Wicklow     1       2011
Ballymore Eustace Kildare     1       2010
         Maynooth Kildare     1       2009


---
## 1.14 GPU Check and Environment Info

In [19]:
import torch

print("Environment Info:")
print("   Python:", sys.version.split()[0])
print("   PyTorch:", torch.__version__)
print("   CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("   GPU:", torch.cuda.get_device_name(0))
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("   GPU Memory: {:.1f} GB".format(gpu_mem))
else:
    print("   No GPU detected.")
    print("   Switch to GPU for Section 3+: Runtime > Change runtime type > GPU")


Environment Info:
   Python: 3.12.13
   PyTorch: 2.10.0+cu128
   CUDA available: True
   GPU: NVIDIA A100-SXM4-40GB
   GPU Memory: 39.5 GB


---
## Section 1 Complete!

### What is now ready:
1. All dependencies installed
2. Google Drive mounted with folder structure
3. Cache manager for saving/loading expensive outputs
4. WikiTableQuestions dataset loaded and processed
5. Tables converted to DataFrames with clean column names
6. SQLite databases created for SQL execution
7. SQL execution engine ready
8. Evaluation framework (execution accuracy, SQL validity)
9. Query complexity classifier (Simple/Medium/Complex)
10. Stratified eval subset (500 examples for quick testing)
11. Schema formatter for LLM prompts
12. GPU environment verified

### Cached on Drive (will not re-run next session):
- data/processed_train.pkl
- data/processed_test.pkl
- data/train_tables.db
- data/test_tables.db
- data/complexity_labels.json
- data/eval_subset_500.json

### Next: Section 2 - Template-Based SQL Baseline

## Section 2: Template-Based SQL Baseline (Non-LLM)

This is our first baseline — a rule-based system that converts natural language questions to SQL using keyword matching and patterns. No LLM involved. This sets the floor for comparison.

In [20]:
def template_based_sql(question, table_info):
    """
    Generate SQL from natural language using keyword matching rules.
    No LLM — purely rule-based.
    """
    q = question.lower().strip()
    table_name = "t"  # placeholder, will be replaced by execute_sql
    columns = table_info['clean_cols']
    schema = table_info['schema']

    # Try to find which column the question is about
    def find_relevant_column(q, columns, schema):
        """Match question words to column names."""
        best_col = None
        best_score = 0
        for i, col_info in enumerate(schema):
            score = 0
            orig_name = col_info['original_name'].lower()
            clean_name = col_info['clean_name'].lower()
            # Check if original column name appears in question
            for word in orig_name.split():
                if len(word) > 2 and word in q:
                    score += 2
            for word in clean_name.split('_'):
                if len(word) > 2 and word in q:
                    score += 1
            if score > best_score:
                best_score = score
                best_col = columns[i]
        return best_col, best_score

    # Find the column most relevant to the question
    target_col, col_score = find_relevant_column(q, columns, schema)

    # Find a filter column and value (look for quoted values or proper nouns)
    def find_filter(q, columns, schema, table_info):
        """Try to find a WHERE clause from the question."""
        df = table_info['df']
        best_filter_col = None
        best_filter_val = None
        best_match_len = 0

        for col in columns:
            try:
                unique_vals = df[col].dropna().astype(str).unique()
                for val in unique_vals:
                    val_lower = val.lower()
                    if len(val_lower) > 2 and val_lower in q:
                        if len(val_lower) > best_match_len:
                            best_match_len = len(val_lower)
                            best_filter_col = col
                            best_filter_val = val
            except:
                continue

        return best_filter_col, best_filter_val

    filter_col, filter_val = find_filter(q, columns, schema, table_info)

    # Determine query type based on keywords
    sql = None

    # COUNT queries
    if any(kw in q for kw in ['how many', 'count', 'number of', 'total number']):
        if filter_col and filter_val:
            sql = f'SELECT COUNT(*) FROM t WHERE "{filter_col}" = \'{filter_val}\''
        elif target_col:
            sql = f'SELECT COUNT(DISTINCT "{target_col}") FROM t'
        else:
            sql = 'SELECT COUNT(*) FROM t'

    # SUM queries
    elif any(kw in q for kw in ['total', 'sum', 'combined', 'altogether']):
        if target_col:
            sql = f'SELECT SUM("{target_col}") FROM t'
        else:
            # Try to find a numeric column
            for col_info in schema:
                if 'int' in col_info['dtype'] or 'float' in col_info['dtype']:
                    sql = f'SELECT SUM("{col_info["clean_name"]}") FROM t'
                    break

    # AVERAGE queries
    elif any(kw in q for kw in ['average', 'avg', 'mean']):
        if target_col:
            sql = f'SELECT AVG("{target_col}") FROM t'
        else:
            for col_info in schema:
                if 'int' in col_info['dtype'] or 'float' in col_info['dtype']:
                    sql = f'SELECT AVG("{col_info["clean_name"]}") FROM t'
                    break

    # MAX/HIGHEST queries
    elif any(kw in q for kw in ['most', 'highest', 'largest', 'greatest', 'maximum', 'max', 'best', 'top']):
        if target_col:
            if filter_col and filter_col != target_col:
                sql = f'SELECT "{filter_col}" FROM t ORDER BY "{target_col}" DESC LIMIT 1'
            else:
                sql = f'SELECT MAX("{target_col}") FROM t'
        else:
            sql = f'SELECT * FROM t ORDER BY "{columns[-1]}" DESC LIMIT 1'

    # MIN/LOWEST queries
    elif any(kw in q for kw in ['least', 'lowest', 'smallest', 'minimum', 'min', 'worst', 'fewest']):
        if target_col:
            if filter_col and filter_col != target_col:
                sql = f'SELECT "{filter_col}" FROM t ORDER BY "{target_col}" ASC LIMIT 1'
            else:
                sql = f'SELECT MIN("{target_col}") FROM t'
        else:
            sql = f'SELECT * FROM t ORDER BY "{columns[-1]}" ASC LIMIT 1'

    # LAST/LATEST queries
    elif any(kw in q for kw in ['last', 'latest', 'most recent', 'final']):
        if target_col:
            sql = f'SELECT "{target_col}" FROM t ORDER BY ROWID DESC LIMIT 1'
        elif filter_col:
            sql = f'SELECT "{filter_col}" FROM t ORDER BY ROWID DESC LIMIT 1'
        else:
            sql = f'SELECT * FROM t ORDER BY ROWID DESC LIMIT 1'

    # FIRST/EARLIEST queries
    elif any(kw in q for kw in ['first', 'earliest', 'initial', 'beginning']):
        if target_col:
            sql = f'SELECT "{target_col}" FROM t ORDER BY ROWID ASC LIMIT 1'
        elif filter_col:
            sql = f'SELECT "{filter_col}" FROM t ORDER BY ROWID ASC LIMIT 1'
        else:
            sql = f'SELECT * FROM t ORDER BY ROWID ASC LIMIT 1'

    # Simple lookup with filter
    elif filter_col and filter_val and target_col:
        sql = f'SELECT "{target_col}" FROM t WHERE "{filter_col}" = \'{filter_val}\''

    # Fallback: just select the most relevant column
    elif target_col:
        sql = f'SELECT "{target_col}" FROM t LIMIT 1'

    # Ultimate fallback
    if sql is None:
        sql = f'SELECT "{columns[0]}" FROM t LIMIT 1'

    return sql


print("Template SQL generator defined!")

Template SQL generator defined!


##  Test Template on Examples:

In [21]:
# Test on a few examples to see what SQL it generates
print("Template SQL Examples:")
print("=" * 70)

for i in range(5):
    ex = test_examples[i]
    table_info = test_tables[ex['table_id']]
    sql = template_based_sql(ex['question'], table_info)
    result, success, error = execute_sql(sql, ex['table_id'], TEST_DB_PATH)

    print(f"\nQ: {ex['question']}")
    print(f"   Generated SQL: {sql}")
    print(f"   Result: {result[:3] if result else 'empty'}")
    print(f"   Gold Answer: {ex['answers']}")
    match, _ = check_answer(result, ex['answers'])
    print(f"   Correct: {match}")
    print("-" * 70)

Template SQL Examples:

Q: which team won previous to crettyard?
   Generated SQL: SELECT "team" FROM t WHERE "team" = 'Crettyard'
   Result: ['Crettyard']
   Gold Answer: ['Wolfe Tones']
   Correct: False
----------------------------------------------------------------------

Q: how many more passengers flew to los angeles than to saskatoon from manzanillo airport in 2013?
   Generated SQL: SELECT COUNT(DISTINCT "passengers") FROM t
   Result: empty
   Gold Answer: ['12,467']
   Correct: False
----------------------------------------------------------------------

Q: after winning on four credits with a full house, what is your payout?
   Generated SQL: SELECT "col_1_credit" FROM t WHERE "hand" = 'Full house'
   Result: ['8.0']
   Gold Answer: ['32']
   Correct: False
----------------------------------------------------------------------

Q: which players played the same position as ardo kreek?
   Generated SQL: SELECT "player" FROM t WHERE "player" = 'Ardo Kreek'
   Result: ['Ardo Kr

### Run Template Baseline on Eval Subset:

In [22]:
# Run template baseline on the 500-example eval subset
print("Running Template SQL baseline on eval subset...")

template_predictions = []
for ex in tqdm(eval_subset, desc="Template SQL"):
    table_info = test_tables[ex['table_id']]

    start_time = time.time()
    sql = template_based_sql(ex['question'], table_info)
    latency = time.time() - start_time

    template_predictions.append({
        'id': ex['id'],
        'sql': sql,
        'latency': latency
    })

print(f"Generated {len(template_predictions)} SQL queries")

Running Template SQL baseline on eval subset...


Template SQL:   0%|          | 0/499 [00:00<?, ?it/s]

Generated 499 SQL queries


### Evaluate Template Baseline

In [23]:
# Evaluate
template_metrics, template_details = evaluate_predictions(
    template_predictions, eval_subset, TEST_DB_PATH, test_tables
)

print()
print("=" * 50)
print("TEMPLATE SQL BASELINE RESULTS")
print("=" * 50)
print(f"   Execution Accuracy: {template_metrics['execution_accuracy']:.1%}")
print(f"   SQL Validity Rate:  {template_metrics['sql_validity_rate']:.1%}")
print(f"   Correct: {template_metrics['correct_count']} / {template_metrics['total']}")
print(f"   Avg Latency: {template_metrics['avg_latency']*1000:.1f} ms")
print(f"   Errors: {template_metrics['error_count']}")
print("=" * 50)

Evaluating:   0%|          | 0/499 [00:00<?, ?it/s]


TEMPLATE SQL BASELINE RESULTS
   Execution Accuracy: 11.6%
   SQL Validity Rate:  62.5%
   Correct: 58 / 499
   Avg Latency: 1.4 ms
   Errors: 187


In [24]:
# Debug: check a few examples
for i in range(5):
    ex = eval_subset[i]
    table_info = test_tables[ex['table_id']]
    sql = template_based_sql(ex['question'], table_info)
    result, success, error = execute_sql(sql, ex['table_id'], TEST_DB_PATH)

    print(f"Q: {ex['question']}")
    print(f"   Gold answers (raw): {repr(ex['answers'])}")
    print(f"   SQL result (raw): {repr(result)}")
    print(f"   SQL: {sql}")
    print(f"   Success: {success}")
    print("-" * 60)

Q: how many consecutive times did they play in tier 4?
   Gold answers (raw): ['6']
   SQL result (raw): ['6']
   SQL: SELECT COUNT(*) FROM t WHERE "level" = 'Tier 4'
   Success: True
------------------------------------------------------------
Q: how long was it between losing streak almbum and gnv fla in years.
   Gold answers (raw): ['11']
   SQL result (raw): ['1997', '1997']
   SQL: SELECT "year" FROM t WHERE "album" = 'Losing Streak'
   Success: True
------------------------------------------------------------
Q: who scored the least?
   Gold answers (raw): ['Wicklow']
   SQL result (raw): []
   SQL: SELECT MIN("score") FROM t
   Success: False
------------------------------------------------------------
Q: previous to tom andrews who was elected?
   Gold answers (raw): ['Gordon Anderson']
   SQL result (raw): ['1949.0']
   SQL: SELECT "first_elected" FROM t WHERE "member" = 'Tom Andrews'
   Success: True
------------------------------------------------------------
Q: what is the

### Breakdown by Complexity

In [25]:
# Breakdown by complexity
print("Results by Complexity:")
print("-" * 50)

for level in ['simple', 'medium', 'complex']:
    level_details = [d for d in template_details if complexity_labels.get(d['id']) == level]
    if level_details:
        correct = sum(1 for d in level_details if d['correct'])
        total = len(level_details)
        print(f"   {level.capitalize():10s}: {correct}/{total} = {correct/total:.1%}")

Results by Complexity:
--------------------------------------------------
   Simple    : 42/279 = 15.1%
   Medium    : 16/180 = 8.9%
   Complex   : 0/40 = 0.0%


### Error Analysis

In [26]:
# Error analysis - what types of errors does template SQL make?
from collections import Counter

error_types = []
for d in template_details:
    if not d['correct']:
        if d['match_type'] == 'sql_error':
            error_types.append('SQL Error')
        elif d['match_type'] == 'empty':
            error_types.append('Empty Result')
        else:
            error_types.append('Wrong Answer')

error_counts = Counter(error_types)
print("Error Breakdown:")
print("-" * 40)
for error_type, count in error_counts.most_common():
    print(f"   {error_type:15s}: {count} ({count/len(template_details):.1%})")

Error Breakdown:
----------------------------------------
   Wrong Answer   : 254 (50.9%)
   SQL Error      : 187 (37.5%)


### Save Results to Drive

In [27]:
# Save predictions and metrics to Drive
cache.save_json(template_predictions, 'results/baseline_template', 'predictions.json')
cache.save_json(template_metrics, 'results/baseline_template', 'metrics.json')
cache.save_json(template_details, 'results/baseline_template', 'details.json')

print()
print("Section 2 complete! Template baseline saved to Drive.")
print("This is our floor - every LLM approach should beat this.")

Saved to Drive: results/baseline_template/predictions.json
Saved to Drive: results/baseline_template/metrics.json
Saved to Drive: results/baseline_template/details.json

Section 2 complete! Template baseline saved to Drive.
This is our floor - every LLM approach should beat this.


## Section 3: Zero-Shot LLM Baseline (Qwen2.5-Coder-7B)

First LLM approach — give the model the table schema and question, ask it to generate SQL with no examples. This tests the model's raw ability to understand natural language and produce valid SQL.

Model: **Qwen2.5-Coder-7B-Instruct** — a code-specialized LLM optimized for code generation tasks including SQL.

### Load Qwen2.5-Coder-7B-Instruct

In [28]:
!pip install "bitsandbytes>=0.46.1"
import bitsandbytes
print("bitsandbytes version:", bitsandbytes.__version__)

  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.2 MB/s eta 0:00:00
bitsandbytes version: 0.49.2


In [29]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME} in 4-bit quantization...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

print()
print("Model loaded!")
print(f"   Model: {MODEL_NAME}")
print(f"   GPU Memory Used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")


Loading Qwen/Qwen2.5-Coder-7B-Instruct in 4-bit quantization...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded!
   Model: Qwen/Qwen2.5-Coder-7B-Instruct
   GPU Memory Used: 5.2 GB


### Define Generation Function

In [30]:
def generate_sql(prompt, model, tokenizer, max_new_tokens=256):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    start_time = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.time() - start_time

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    response = tokenizer.decode(output_ids, skip_special_tokens=True)

    return response, latency


print("Generation function defined!")


Generation function defined!


### Define Zero-Shot Prompt Template

In [31]:
def build_zeroshot_prompt(question, table_info):
    """
    Build a zero-shot prompt for SQL generation.
    Gives the model the table schema and asks for SQL.
    """
    schema_text = format_schema(table_info, include_samples=True, max_sample_rows=3)

    prompt = f"""You are a SQL expert. Given a table and a question, generate a SQLite query to answer the question.

{schema_text}

Question: {question}

Rules:
- Return ONLY the SQL query, nothing else
- Use the exact column names provided above
- The table name is "t"
- Use SQLite syntax

SQL:"""

    return prompt


# Test the prompt
sample_ex = test_examples[0]
sample_table = test_tables[sample_ex['table_id']]
sample_prompt = build_zeroshot_prompt(sample_ex['question'], sample_table)
print("Sample Zero-Shot Prompt:")
print("=" * 60)
print(sample_prompt)
print("=" * 60)

Sample Zero-Shot Prompt:
You are a SQL expert. Given a table and a question, generate a SQLite query to answer the question.

Table has 9 rows and 4 columns.
Columns:
  - team (text) | Examples: Greystones, Ballymore Eustace, Maynooth
  - county (text) | Examples: Wicklow, Kildare, Kildare
  - wins (numeric) | Examples: 1, 1, 1
  - years_won (numeric) | Examples: 2011, 2010, 2009

Sample rows (first 3):
             team  county  wins  years_won
       Greystones Wicklow     1       2011
Ballymore Eustace Kildare     1       2010
         Maynooth Kildare     1       2009

Question: which team won previous to crettyard?

Rules:
- Return ONLY the SQL query, nothing else
- Use the exact column names provided above
- The table name is "t"
- Use SQLite syntax

SQL:


### Test on a Few Examples

In [32]:
# Test on 5 examples to make sure everything works
print("Testing zero-shot generation on 5 examples...")
print("=" * 70)

for i in range(5):
    ex = test_examples[i]
    table_info = test_tables[ex['table_id']]
    prompt = build_zeroshot_prompt(ex['question'], table_info)

    response, latency = generate_sql(prompt, model, tokenizer)

    # Extract SQL from response (take first line that looks like SQL)
    sql = response.strip().split('\n')[0].strip()
    # Remove markdown code blocks if present
    sql = sql.replace('```sql', '').replace('```', '').strip()

    result, success, error = execute_sql(sql, ex['table_id'], TEST_DB_PATH)
    match, _ = check_answer(result, ex['answers'])

    print(f"\nQ: {ex['question']}")
    print(f"   Generated SQL: {sql}")
    print(f"   Result: {result[:3] if result else 'empty/error'}")
    print(f"   Gold Answer: {ex['answers']}")
    print(f"   Correct: {match} | Latency: {latency:.2f}s")
    print("-" * 70)

Testing zero-shot generation on 5 examples...

Q: which team won previous to crettyard?
   Generated SQL: 
   Result: empty/error
   Gold Answer: ['Wolfe Tones']
   Correct: False | Latency: 3.91s
----------------------------------------------------------------------

Q: how many more passengers flew to los angeles than to saskatoon from manzanillo airport in 2013?
   Generated SQL: 
   Result: empty/error
   Gold Answer: ['12,467']
   Correct: False | Latency: 3.63s
----------------------------------------------------------------------

Q: after winning on four credits with a full house, what is your payout?
   Generated SQL: 
   Result: empty/error
   Gold Answer: ['32']
   Correct: False | Latency: 1.56s
----------------------------------------------------------------------

Q: which players played the same position as ardo kreek?
   Generated SQL: 
   Result: empty/error
   Gold Answer: ['Siim Ennemuist', 'Andri Aganits']
   Correct: False | Latency: 1.76s
-------------------------

### SQL Extraction Helper

In [33]:
def extract_sql_from_response(response):
    """
    Extract clean SQL from model response.
    Handles various output formats (markdown blocks, explanations, etc.)
    """
    text = response.strip()

    # Remove markdown code blocks
    if '```sql' in text:
        match = re.search(r'```sql\s*(.*?)\s*```', text, re.DOTALL)
        if match:
            return match.group(1).strip()

    if '```' in text:
        match = re.search(r'```\s*(.*?)\s*```', text, re.DOTALL)
        if match:
            return match.group(1).strip()

    # Take first line that starts with SELECT, WITH, etc.
    for line in text.split('\n'):
        line = line.strip()
        if line.upper().startswith(('SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE')):
            # Collect multi-line SQL
            sql_lines = [line]
            remaining = text.split('\n')
            start_idx = remaining.index(line) if line in remaining else -1
            if start_idx >= 0:
                for next_line in remaining[start_idx + 1:]:
                    next_line = next_line.strip()
                    if next_line and not next_line.startswith(('--', '#', 'Note', 'This', 'The', 'Here')):
                        sql_lines.append(next_line)
                        if next_line.endswith(';'):
                            break
                    else:
                        break
            return ' '.join(sql_lines).rstrip(';').strip()

    # Fallback: return first non-empty line
    for line in text.split('\n'):
        if line.strip():
            return line.strip()

    return text


print("SQL extraction helper defined!")

SQL extraction helper defined!


Run Zero-Shot on Eval Subset

In [34]:
# Run zero-shot on the 500-example eval subset
if cache.exists('results/baseline_zeroshot', 'predictions.json'):
    print("Loading zero-shot results from Drive cache...")
    zeroshot_predictions = cache.load_json('results/baseline_zeroshot', 'predictions.json')
else:
    print("Running zero-shot Qwen2.5-Coder-7B on eval subset...")

    zeroshot_predictions = []
    errors_during_generation = 0

    for i, ex in enumerate(tqdm(eval_subset, desc="Zero-shot")):
        table_info = test_tables[ex['table_id']]
        prompt = build_zeroshot_prompt(ex['question'], table_info)

        try:
            response, latency = generate_sql(prompt, model, tokenizer)
            sql = extract_sql_from_response(response)
        except Exception as e:
            sql = "SELECT 1"
            latency = 0
            errors_during_generation += 1

        zeroshot_predictions.append({
            'id': ex['id'],
            'sql': sql,
            'latency': latency
        })

        if (i + 1) % 100 == 0:
            print(f"   Completed {i+1}/{len(eval_subset)} examples")

    cache.save_json(zeroshot_predictions, 'results/baseline_zeroshot', 'predictions.json')
    print("Zero-shot predictions saved to Drive!")

Loading zero-shot results from Drive cache...
Loaded from Drive: results/baseline_zeroshot/predictions.json


### Evaluate Zero-Shot

In [35]:
# Evaluate zero-shot results
zeroshot_metrics, zeroshot_details = evaluate_predictions(
    zeroshot_predictions, eval_subset, TEST_DB_PATH, test_tables
)

print()
print("=" * 50)
print("ZERO-SHOT QWEN2.5-CODER-7B RESULTS")
print("=" * 50)
print(f"   Execution Accuracy: {zeroshot_metrics['execution_accuracy']:.1%}")
print(f"   SQL Validity Rate:  {zeroshot_metrics['sql_validity_rate']:.1%}")
print(f"   Correct: {zeroshot_metrics['correct_count']} / {zeroshot_metrics['total']}")
print(f"   Avg Latency: {zeroshot_metrics['avg_latency']:.2f}s")
print(f"   Errors: {zeroshot_metrics['error_count']}")
print("=" * 50)

print()
print("Comparison with Template Baseline:")
print(f"   Template:  {template_metrics['execution_accuracy']:.1%}")
print(f"   Zero-shot: {zeroshot_metrics['execution_accuracy']:.1%}")
improvement = zeroshot_metrics['execution_accuracy'] - template_metrics['execution_accuracy']
print(f"   Improvement: +{improvement:.1%}")


Evaluating:   0%|          | 0/499 [00:00<?, ?it/s]


ZERO-SHOT QWEN2.5-CODER-7B RESULTS
   Execution Accuracy: 44.1%
   SQL Validity Rate:  83.0%
   Correct: 220 / 499
   Avg Latency: 1.94s
   Errors: 85

Comparison with Template Baseline:
   Template:  11.6%
   Zero-shot: 44.1%
   Improvement: +32.5%


### Complexity Breakdown

In [36]:
# Breakdown by complexity
print("Zero-Shot Results by Complexity:")
print("-" * 50)

for level in ['simple', 'medium', 'complex']:
    level_details = [d for d in zeroshot_details if complexity_labels.get(d['id']) == level]
    if level_details:
        correct = sum(1 for d in level_details if d['correct'])
        total = len(level_details)
        print(f"   {level.capitalize():10s}: {correct}/{total} = {correct/total:.1%}")

Zero-Shot Results by Complexity:
--------------------------------------------------
   Simple    : 131/279 = 47.0%
   Medium    : 73/180 = 40.6%
   Complex   : 16/40 = 40.0%


### Error Analysis

In [37]:
# Error analysis
error_types = []
for d in zeroshot_details:
    if not d['correct']:
        if d['match_type'] == 'sql_error':
            error_types.append('SQL Error')
        elif d['match_type'] == 'empty':
            error_types.append('Empty Result')
        else:
            error_types.append('Wrong Answer')

error_counts = Counter(error_types)
print("Zero-Shot Error Breakdown:")
print("-" * 40)
for error_type, count in error_counts.most_common():
    print(f"   {error_type:15s}: {count} ({count/len(zeroshot_details):.1%})")

Zero-Shot Error Breakdown:
----------------------------------------
   Wrong Answer   : 149 (29.9%)
   SQL Error      : 85 (17.0%)
   Empty Result   : 45 (9.0%)


### Save Results

In [38]:
# Save to Drive
cache.save_json(zeroshot_predictions, 'results/baseline_zeroshot', 'predictions.json')
cache.save_json(zeroshot_metrics, 'results/baseline_zeroshot', 'metrics.json')
cache.save_json(zeroshot_details, 'results/baseline_zeroshot', 'details.json')

print()
print("Section 3 complete! Zero-shot results saved to Drive.")
print("Next: Section 4 - Few-shot & Chain-of-Thought prompting")

Saved to Drive: results/baseline_zeroshot/predictions.json
Saved to Drive: results/baseline_zeroshot/metrics.json
Saved to Drive: results/baseline_zeroshot/details.json

Section 3 complete! Zero-shot results saved to Drive.
Next: Section 4 - Few-shot & Chain-of-Thought prompting


## Section 4: Few-Shot, Chain-of-Thought & Self-Consistency

Three prompting strategies that build on zero-shot by giving the model more guidance:
- **Few-shot:** Show solved examples before the question
- **Chain-of-Thought (CoT):** Ask the model to reason step-by-step
- **Self-Consistency:** Run multiple times, take majority vote

### Build Few-Shot Examples

In [39]:
# Build few-shot examples from training data
def get_fewshot_examples(train_examples, train_tables, n=5):
    """Select diverse few-shot examples from training data."""
    examples = []
    seen_types = set()

    # Pick examples that cover different query types
    keywords_groups = [
        ['how many', 'count', 'total'],
        ['most', 'highest', 'largest', 'maximum'],
        ['least', 'lowest', 'smallest', 'minimum'],
        ['who', 'which', 'what'],
        ['average', 'mean', 'sum']
    ]

    for ex in train_examples:
        if len(examples) >= n:
            break

        q = ex['question'].lower()
        table_info = train_tables.get(ex['table_id'])
        if table_info is None:
            continue

        # Try to find which type this question belongs to
        q_type = 'general'
        for i, keywords in enumerate(keywords_groups):
            if any(kw in q for kw in keywords):
                q_type = str(i)
                break

        if q_type not in seen_types or len(examples) < 3:
            seen_types.add(q_type)

            # Generate a simple SQL for this example
            schema = format_schema(table_info, include_samples=False)

            examples.append({
                'question': ex['question'],
                'answers': ex['answers'],
                'schema': schema,
                'table_id': ex['table_id']
            })

    return examples

fewshot_examples = get_fewshot_examples(train_examples, train_tables, n=5)
print(f"Selected {len(fewshot_examples)} few-shot examples")
for i, ex in enumerate(fewshot_examples):
    print(f"  {i+1}. {ex['question'][:80]}...")

Selected 5 few-shot examples
  1. what was the last year where this team was a part of the usl a-league?...
  2. in what city did piotr's last 1st place finish occur?...
  3. who was the opponent in the first game of the season?...
  4. how many people stayed at least 3 years in office?...
  5. which album released by the band schnell fenster produced the most singles appea...


### Generate Few-Shot SQL for Examples

In [40]:
# Generate correct SQL for few-shot examples by running them through the model
# and manually selecting the ones that produce correct answers
print("Generating SQL for few-shot examples...")

fewshot_pairs = []
for ex in fewshot_examples:
    table_info = train_tables[ex['table_id']]
    prompt = build_zeroshot_prompt(ex['question'], table_info)
    response, _ = generate_sql(prompt, model, tokenizer)
    sql = extract_sql_from_response(response)

    # Execute and check
    result, success, error = execute_sql(sql, ex['table_id'], TRAIN_DB_PATH)
    match, _ = check_answer(result, ex['answers'])

    if match and success:
        fewshot_pairs.append({
            'question': ex['question'],
            'sql': sql,
            'schema': ex['schema'],
            'answer': ex['answers'][0]
        })
        print(f"  OK: {ex['question'][:60]}...")
    else:
        print(f"  SKIP (wrong answer): {ex['question'][:60]}...")

print(f"\n{len(fewshot_pairs)} usable few-shot examples")

# If we don't have enough correct examples, add manual ones
if len(fewshot_pairs) < 3:
    print("Adding manual few-shot examples...")
    fewshot_pairs = [
        {
            'question': 'How many rows are in the table?',
            'sql': 'SELECT COUNT(*) FROM t',
            'schema': 'Table with multiple rows and columns.',
            'answer': 'varies'
        },
        {
            'question': 'What is the maximum value in the score column?',
            'sql': 'SELECT MAX(score) FROM t',
            'schema': 'Table with a score column.',
            'answer': 'varies'
        },
        {
            'question': 'Which name appears in the first row?',
            'sql': 'SELECT name FROM t LIMIT 1',
            'schema': 'Table with a name column.',
            'answer': 'varies'
        }
    ]
    print(f"Using {len(fewshot_pairs)} manual examples")

Generating SQL for few-shot examples...
  SKIP (wrong answer): what was the last year where this team was a part of the usl...
  OK: in what city did piotr's last 1st place finish occur?...
  SKIP (wrong answer): who was the opponent in the first game of the season?...
  SKIP (wrong answer): how many people stayed at least 3 years in office?...
  SKIP (wrong answer): which album released by the band schnell fenster produced th...

1 usable few-shot examples
Adding manual few-shot examples...
Using 3 manual examples


### Few-Shot Prompt Template

In [41]:
def build_fewshot_prompt(question, table_info, fewshot_pairs):
    """Build a few-shot prompt with solved examples."""
    schema_text = format_schema(table_info, include_samples=True, max_sample_rows=3)

    # Build examples section
    examples_text = ""
    for i, pair in enumerate(fewshot_pairs[:3]):
        examples_text += f"""
Example {i+1}:
Question: {pair['question']}
SQL: {pair['sql']}
"""

    prompt = f"""You are a SQL expert. Given a table and a question, generate a SQLite query to answer the question.

Here are some examples of correct SQL queries:
{examples_text}
Now answer this question:

{schema_text}

Question: {question}

Rules:
- Return ONLY the SQL query, nothing else
- Use the exact column names provided above
- The table name is "t"
- Use SQLite syntax

SQL:"""

    return prompt

print("Few-shot prompt template defined!")

Few-shot prompt template defined!


### Run Few-Shot on Eval Subset

In [42]:
# Run few-shot on eval subset
if cache.exists('results/fewshot', 'predictions.json'):
    print("Loading few-shot results from Drive cache...")
    fewshot_predictions = cache.load_json('results/fewshot', 'predictions.json')
else:
    print("Running few-shot Qwen2.5-Coder-7B on eval subset...")
    print("This will take approximately 30-60 minutes...")

    fewshot_predictions = []

    for i, ex in enumerate(tqdm(eval_subset, desc="Few-shot")):
        table_info = test_tables[ex['table_id']]
        prompt = build_fewshot_prompt(ex['question'], table_info, fewshot_pairs)

        try:
            response, latency = generate_sql(prompt, model, tokenizer)
            sql = extract_sql_from_response(response)
        except Exception as e:
            sql = "SELECT 1"
            latency = 0

        fewshot_predictions.append({
            'id': ex['id'],
            'sql': sql,
            'latency': latency
        })

        if (i + 1) % 100 == 0:
            print(f"   Completed {i+1}/{len(eval_subset)} examples")

    cache.save_json(fewshot_predictions, 'results/fewshot', 'predictions.json')
    print("Few-shot predictions saved to Drive!")

Loading few-shot results from Drive cache...
Loaded from Drive: results/fewshot/predictions.json


### Evaluate Few-Shot

In [43]:
fewshot_metrics, fewshot_details = evaluate_predictions(
    fewshot_predictions, eval_subset, TEST_DB_PATH, test_tables
)

print()
print("=" * 50)
print("FEW-SHOT QWEN2.5-CODER-7B RESULTS")
print("=" * 50)
print(f"   Execution Accuracy: {fewshot_metrics['execution_accuracy']:.1%}")
print(f"   SQL Validity Rate:  {fewshot_metrics['sql_validity_rate']:.1%}")
print(f"   Correct: {fewshot_metrics['correct_count']} / {fewshot_metrics['total']}")
print(f"   Avg Latency: {fewshot_metrics['avg_latency']:.2f}s")
print("=" * 50)

print()
print("Comparison:")
print(f"   Template:  {template_metrics['execution_accuracy']:.1%}")
print(f"   Zero-shot: {zeroshot_metrics['execution_accuracy']:.1%}")
print(f"   Few-shot:  {fewshot_metrics['execution_accuracy']:.1%}")

cache.save_json(fewshot_metrics, 'results/fewshot', 'metrics.json')
cache.save_json(fewshot_details, 'results/fewshot', 'details.json')

Evaluating:   0%|          | 0/499 [00:00<?, ?it/s]


FEW-SHOT QWEN2.5-CODER-7B RESULTS
   Execution Accuracy: 45.7%
   SQL Validity Rate:  87.4%
   Correct: 228 / 499
   Avg Latency: 1.89s

Comparison:
   Template:  11.6%
   Zero-shot: 44.1%
   Few-shot:  45.7%
Saved to Drive: results/fewshot/metrics.json
Saved to Drive: results/fewshot/details.json


### Few-Shot Complexity Breakdown

In [44]:
print("Few-Shot Results by Complexity:")
print("-" * 50)
for level in ['simple', 'medium', 'complex']:
    level_details = [d for d in fewshot_details if complexity_labels.get(d['id']) == level]
    if level_details:
        correct = sum(1 for d in level_details if d['correct'])
        total = len(level_details)
        print(f"   {level.capitalize():10s}: {correct}/{total} = {correct/total:.1%}")

Few-Shot Results by Complexity:
--------------------------------------------------
   Simple    : 139/279 = 49.8%
   Medium    : 76/180 = 42.2%
   Complex   : 13/40 = 32.5%


### Chain-of-Thought (CoT) Prompting

Ask the model to reason step-by-step before generating SQL. This helps with complex queries that require multi-step logic.

### CoT Prompt Template

In [45]:
def build_cot_prompt(question, table_info):
    """Build a Chain-of-Thought prompt that asks for step-by-step reasoning."""
    schema_text = format_schema(table_info, include_samples=True, max_sample_rows=3)

    prompt = f"""You are a SQL expert. Given a table and a question, think step-by-step to generate a SQLite query.

{schema_text}

Question: {question}

Think step-by-step:
1. What is the question asking for?
2. Which columns are relevant?
3. What SQL operations are needed (SELECT, WHERE, GROUP BY, ORDER BY, etc.)?
4. Write the final SQL query.

After your reasoning, write the final SQL query on a line starting with "SQL:".

SQL:"""

    return prompt

print("CoT prompt template defined!")

CoT prompt template defined!


### Run CoT on Eval Subset

In [46]:
# Run Chain-of-Thought on eval subset
if cache.exists('results/cot', 'predictions.json'):
    print("Loading CoT results from Drive cache...")
    cot_predictions = cache.load_json('results/cot', 'predictions.json')
else:
    print("Running Chain-of-Thought on eval subset...")

    cot_predictions = []

    for i, ex in enumerate(tqdm(eval_subset, desc="CoT")):
        table_info = test_tables[ex['table_id']]
        prompt = build_cot_prompt(ex['question'], table_info)

        try:
            response, latency = generate_sql(prompt, model, tokenizer, max_new_tokens=256)
            sql = extract_sql_from_response(response)
        except Exception as e:
            sql = "SELECT 1"
            latency = 0

        cot_predictions.append({
            'id': ex['id'],
            'sql': sql,
            'latency': latency
        })

        if (i + 1) % 100 == 0:
            print(f"   Completed {i+1}/{len(eval_subset)} examples")

    cache.save_json(cot_predictions, 'results/cot', 'predictions.json')
    print("CoT predictions saved to Drive!")

Loading CoT results from Drive cache...
Loaded from Drive: results/cot/predictions.json


### Evaluate CoT

In [47]:
cot_metrics, cot_details = evaluate_predictions(
    cot_predictions, eval_subset, TEST_DB_PATH, test_tables
)

print()
print("=" * 50)
print("CHAIN-OF-THOUGHT RESULTS")
print("=" * 50)
print(f"   Execution Accuracy: {cot_metrics['execution_accuracy']:.1%}")
print(f"   SQL Validity Rate:  {cot_metrics['sql_validity_rate']:.1%}")
print(f"   Correct: {cot_metrics['correct_count']} / {cot_metrics['total']}")
print(f"   Avg Latency: {cot_metrics['avg_latency']:.2f}s")
print("=" * 50)

print()
print("Comparison:")
print(f"   Template:  {template_metrics['execution_accuracy']:.1%}")
print(f"   Zero-shot: {zeroshot_metrics['execution_accuracy']:.1%}")
print(f"   Few-shot:  {fewshot_metrics['execution_accuracy']:.1%}")
print(f"   CoT:       {cot_metrics['execution_accuracy']:.1%}")

cache.save_json(cot_metrics, 'results/cot', 'metrics.json')
cache.save_json(cot_details, 'results/cot', 'details.json')

Evaluating:   0%|          | 0/499 [00:00<?, ?it/s]


CHAIN-OF-THOUGHT RESULTS
   Execution Accuracy: 23.0%
   SQL Validity Rate:  41.7%
   Correct: 115 / 499
   Avg Latency: 14.62s

Comparison:
   Template:  11.6%
   Zero-shot: 44.1%
   Few-shot:  45.7%
   CoT:       23.0%
Saved to Drive: results/cot/metrics.json
Saved to Drive: results/cot/details.json


In [48]:
# Debug: check what CoT is actually generating
for i in range(5):
    ex = eval_subset[i]
    table_info = test_tables[ex['table_id']]
    prompt = build_cot_prompt(ex['question'], table_info)
    response, _ = generate_sql(prompt, model, tokenizer, max_new_tokens=256)
    sql = extract_sql_from_response(response)

    print(f"Q: {ex['question'][:60]}")
    print(f"Raw response (first 300 chars): {response[:300]}")
    print(f"Extracted SQL: {sql}")
    print("-" * 60)

Q: how many consecutive times did they play in tier 4?
Raw response (first 300 chars): To answer this question, we need to determine how many consecutive seasons a team played in Tier 4. Here's a step-by-step approach to derive the SQL query:

1. **Identify the relevant column**: The `level` column indicates the tier of the league each season.

2. **Filter the data**: We need to filte
Extracted SQL: WITH Tier4Seasons AS (
    SELECT season,
           season - ROW_NUMBER() OVER (ORDER BY season) AS grp
    FROM your_table_name
    WHERE level = 'Tier 4'
)
SELECT COUNT(*) AS consecutive_tier_4_years
FROM Tier4Seasons
GROUP BY grp;
------------------------------------------------------------
Q: how long was it between losing streak almbum and gnv fla in 
Raw response (first 300 chars): Let's break down the question and the task:

1. **What is the question asking for?**
   The question asks for the time difference in years between two specific albums: "Losing Streak" and "Gnv Fla".

2. **

### Why Did CoT Perform Worse? (Key Finding)

Chain-of-Thought (CoT) prompting actually **hurt** performance, dropping from 44.3% (zero-shot) to 23% (CoT). This might seem surprising, but there's a clear explanation:

**The Problem:** Our model (Qwen2.5-Coder-7B) is a code-specialized model. When we ask it to "think step-by-step," it generates long paragraphs of reasoning text before writing SQL. This causes two issues:

1. **Wrong table names:** Instead of using "t" (our table name), the model starts writing generic SQL with names like "table_name" or "your_table_name" because the reasoning mode makes it behave like a tutorial rather than an executor.

2. **SQL extraction breaks:** Our extraction function struggles to find the actual SQL query buried inside paragraphs of reasoning. Sometimes it grabs reasoning text instead of SQL, or catches only a partial query.

**The Lesson:** Code-specialized models already "think in code" internally — they don't need explicit step-by-step reasoning. Forcing them to reason in natural language actually pulls them away from their strength (direct code generation) into their weakness (verbose explanation). This is consistent with research showing that CoT works best with general-purpose models, not code-specialized ones.

**This is why Self-Consistency (48.1%) worked better** — instead of changing HOW the model thinks, it just runs the same direct generation multiple times and picks the most common answer. It leverages the model's existing code generation strength rather than fighting against it.

### CoT Complexity Breakdown

In [49]:
print("CoT Results by Complexity:")
print("-" * 50)
for level in ['simple', 'medium', 'complex']:
    level_details = [d for d in cot_details if complexity_labels.get(d['id']) == level]
    if level_details:
        correct = sum(1 for d in level_details if d['correct'])
        total = len(level_details)
        print(f"   {level.capitalize():10s}: {correct}/{total} = {correct/total:.1%}")

CoT Results by Complexity:
--------------------------------------------------
   Simple    : 66/279 = 23.7%
   Medium    : 44/180 = 24.4%
   Complex   : 5/40 = 12.5%


### Self-Consistency Prompting

Run the same query multiple times with temperature > 0, collect all SQL outputs, execute each one, and take the majority vote answer. This reduces random errors.

### Self-Consistency Implementation

In [50]:
def generate_sql_with_temperature(prompt, model, tokenizer, max_new_tokens=256, temperature=0.7):
    """Generate SQL with higher temperature for diversity."""
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    response = tokenizer.decode(output_ids, skip_special_tokens=True)
    return response


def self_consistency_predict(question, table_info, table_id, db_path, n_samples=5):
    """Run n_samples predictions and return majority vote answer."""
    prompt = build_zeroshot_prompt(question, table_info)

    results_counter = Counter()
    all_sqls = []

    start_time = time.time()
    for _ in range(n_samples):
        try:
            response = generate_sql_with_temperature(prompt, model, tokenizer)
            sql = extract_sql_from_response(response)
            result, success, error = execute_sql(sql, table_id, db_path)

            if success and result:
                answer = '|'.join(sorted(result))
                results_counter[answer] += 1
                all_sqls.append(sql)
        except:
            continue

    latency = time.time() - start_time

    if results_counter:
        best_answer = results_counter.most_common(1)[0][0]
        best_results = best_answer.split('|')
        # Find the SQL that produced this answer
        best_sql = all_sqls[0] if all_sqls else "SELECT 1"
        return best_sql, best_results, latency
    else:
        return "SELECT 1", [], latency


print("Self-consistency functions defined!")

Self-consistency functions defined!


### Run Self-Consistency on Eval Subset

In [51]:
# Run self-consistency on eval subset (3 samples per query to save time)
if cache.exists('results/self_consistency', 'predictions.json'):
    print("Loading self-consistency results from Drive cache...")
    sc_predictions = cache.load_json('results/self_consistency', 'predictions.json')
    sc_results_list = cache.load_json('results/self_consistency', 'results_list.json')
else:
    print("Running Self-Consistency (3 samples per query) on eval subset...")
    print("This will take approximately 90-120 minutes...")

    # Create results folder
    import os
    os.makedirs(os.path.join(BASE_PATH, 'results/self_consistency'), exist_ok=True)

    sc_predictions = []
    sc_results_list = []

    for i, ex in enumerate(tqdm(eval_subset, desc="Self-Consistency")):
        table_info = test_tables[ex['table_id']]

        best_sql, best_results, latency = self_consistency_predict(
            ex['question'], table_info, ex['table_id'], TEST_DB_PATH, n_samples=3
        )

        sc_predictions.append({
            'id': ex['id'],
            'sql': best_sql,
            'latency': latency
        })

        sc_results_list.append({
            'id': ex['id'],
            'results': best_results
        })

        if (i + 1) % 50 == 0:
            print(f"   Completed {i+1}/{len(eval_subset)} examples")

    cache.save_json(sc_predictions, 'results/self_consistency', 'predictions.json')
    cache.save_json(sc_results_list, 'results/self_consistency', 'results_list.json')
    print("Self-consistency predictions saved to Drive!")

Loading self-consistency results from Drive cache...
Loaded from Drive: results/self_consistency/predictions.json
Loaded from Drive: results/self_consistency/results_list.json


### Evaluate Self-Consistency

In [52]:
# Custom evaluation for self-consistency (uses pre-computed results)
sc_example_lookup = {ex['id']: ex for ex in eval_subset}
sc_correct = 0
sc_total = len(sc_results_list)
sc_details_list = []

for pred, res in zip(sc_predictions, sc_results_list):
    example = sc_example_lookup[pred['id']]
    gold_answers = example['answers']
    predicted_results = res['results']

    match, match_type = check_answer(predicted_results, gold_answers)
    if match:
        sc_correct += 1

    sc_details_list.append({
        'id': pred['id'],
        'question': example['question'],
        'gold_answers': gold_answers,
        'predicted_result': predicted_results,
        'correct': match,
        'match_type': match_type,
        'latency': pred['latency']
    })

sc_accuracy = sc_correct / sc_total
sc_avg_latency = np.mean([d['latency'] for d in sc_details_list])

print()
print("=" * 50)
print("SELF-CONSISTENCY RESULTS (3 samples)")
print("=" * 50)
print(f"   Execution Accuracy: {sc_accuracy:.1%}")
print(f"   Correct: {sc_correct} / {sc_total}")
print(f"   Avg Latency: {sc_avg_latency:.2f}s")
print("=" * 50)

print()
print("Comparison:")
print(f"   Template:          {template_metrics['execution_accuracy']:.1%}")
print(f"   Zero-shot:         {zeroshot_metrics['execution_accuracy']:.1%}")
print(f"   Few-shot:          {fewshot_metrics['execution_accuracy']:.1%}")
print(f"   CoT:               {cot_metrics['execution_accuracy']:.1%}")
print(f"   Self-Consistency:  {sc_accuracy:.1%}")

sc_metrics = {
    'execution_accuracy': sc_accuracy,
    'correct_count': sc_correct,
    'total': sc_total,
    'avg_latency': float(sc_avg_latency)
}
cache.save_json(sc_metrics, 'results/self_consistency', 'metrics.json')
cache.save_json(sc_details_list, 'results/self_consistency', 'details.json')


SELF-CONSISTENCY RESULTS (3 samples)
   Execution Accuracy: 48.1%
   Correct: 240 / 499
   Avg Latency: 6.08s

Comparison:
   Template:          11.6%
   Zero-shot:         44.1%
   Few-shot:          45.7%
   CoT:               23.0%
   Self-Consistency:  48.1%
Saved to Drive: results/self_consistency/metrics.json
Saved to Drive: results/self_consistency/details.json


### Self-Consistency Complexity Breakdown

In [53]:
print("Self-Consistency Results by Complexity:")
print("-" * 50)
for level in ['simple', 'medium', 'complex']:
    level_details = [d for d in sc_details_list if complexity_labels.get(d['id']) == level]
    if level_details:
        correct = sum(1 for d in level_details if d['correct'])
        total = len(level_details)
        print(f"   {level.capitalize():10s}: {correct}/{total} = {correct/total:.1%}")

Self-Consistency Results by Complexity:
--------------------------------------------------
   Simple    : 139/279 = 49.8%
   Medium    : 85/180 = 47.2%
   Complex   : 16/40 = 40.0%


### Section 4 Summary

In [54]:
print("=" * 60)
print("SECTION 4 COMPLETE - PROMPTING STRATEGIES SUMMARY")
print("=" * 60)
print(f"   Template:          {template_metrics['execution_accuracy']:.1%}")
print(f"   Zero-shot:         {zeroshot_metrics['execution_accuracy']:.1%}")
print(f"   Few-shot:          {fewshot_metrics['execution_accuracy']:.1%}")
print(f"   CoT:               {cot_metrics['execution_accuracy']:.1%}")
print(f"   Self-Consistency:  {sc_accuracy:.1%}")
print("=" * 60)
print()
print("All results saved to Drive!")
print("Next: Section 5 - RAG System")

SECTION 4 COMPLETE - PROMPTING STRATEGIES SUMMARY
   Template:          11.6%
   Zero-shot:         44.1%
   Few-shot:          45.7%
   CoT:               23.0%
   Self-Consistency:  48.1%

All results saved to Drive!
Next: Section 5 - RAG System


## Section 5: RAG — Retrieval Augmented Generation

Instead of giving the model the full table schema every time, we use vector search to retrieve only the most relevant columns and sample rows for each question. This gives the model focused context.

**Components:**
- Sentence-Transformers for column embeddings
- FAISS for fast similarity search
- Schema-aware retrieval with sample rows

### Load Embedding Model

In [55]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 106.7 MB/s eta 0:00:00


In [56]:
from sentence_transformers import SentenceTransformer
import faiss

print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Embedding model loaded! Dimension: {embed_model.get_sentence_embedding_dimension()}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded! Dimension: 384


### Build Column Embeddings for All Tables

In [57]:
import os

EMBEDDINGS_PATH = cache.cache_path('embeddings', 'column_embeddings.pkl')

if os.path.exists(EMBEDDINGS_PATH):
    print("Loading column embeddings from Drive cache...")
    col_embeddings_data = cache.load_pickle('embeddings', 'column_embeddings.pkl')
else:
    print("Building column embeddings for all test tables...")

    col_embeddings_data = {}

    for table_id, table_info in tqdm(test_tables.items(), desc="Embedding tables"):
        col_texts = []
        col_names = []

        for col_info in table_info['schema']:
            # Create rich text description for each column
            col_name = col_info['original_name']
            clean_name = col_info['clean_name']
            dtype = 'numeric' if 'int' in col_info['dtype'] or 'float' in col_info['dtype'] else 'text'
            samples = ', '.join(col_info['sample_values'][:3])

            description = f"{col_name} ({dtype}): example values are {samples}"
            col_texts.append(description)
            col_names.append(clean_name)

        # Embed all columns for this table
        embeddings = embed_model.encode(col_texts, normalize_embeddings=True)

        col_embeddings_data[table_id] = {
            'embeddings': embeddings,
            'col_texts': col_texts,
            'col_names': col_names
        }

    cache.save_pickle(col_embeddings_data, 'embeddings', 'column_embeddings.pkl')
    print("Column embeddings saved to Drive!")

print(f"Embeddings ready for {len(col_embeddings_data)} tables")

Loading column embeddings from Drive cache...
Loaded from Drive: embeddings/column_embeddings.pkl
Embeddings ready for 2810 tables


### RAG Retrieval Function

In [58]:
def rag_retrieve(question, table_id, table_info, col_embeddings_data, embed_model, top_k=5):
    """
    Retrieve the most relevant columns for a question using vector similarity.
    Returns a focused schema with only the relevant columns and sample rows.
    """
    if table_id not in col_embeddings_data:
        # Fallback to full schema if embeddings not available
        return format_schema(table_info, include_samples=True, max_sample_rows=3)

    col_data = col_embeddings_data[table_id]
    embeddings = col_data['embeddings']
    col_names = col_data['col_names']
    col_texts = col_data['col_texts']

    # Embed the question
    q_embedding = embed_model.encode([question], normalize_embeddings=True)

    # Find top-k most relevant columns
    num_cols = len(col_names)
    k = min(top_k, num_cols)

    # Build FAISS index for this table
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product = cosine similarity (normalized)
    index.add(embeddings.astype('float32'))

    scores, indices = index.search(q_embedding.astype('float32'), k)

    # Build focused schema with relevant columns
    relevant_cols = [col_names[i] for i in indices[0]]
    relevant_texts = [col_texts[i] for i in indices[0]]
    relevant_scores = scores[0]

    # Format the retrieved schema
    lines = []
    lines.append(f"Table has {table_info['num_rows']} rows and {table_info['num_cols']} columns.")
    lines.append(f"Most relevant columns for your question (ranked by relevance):")

    for i, (col, text, score) in enumerate(zip(relevant_cols, relevant_texts, relevant_scores)):
        lines.append(f"  {i+1}. {text} [relevance: {score:.2f}]")

    # Add all column names so model knows the full schema
    lines.append(f"")
    lines.append(f"All columns: {', '.join(table_info['clean_cols'])}")

    # Add sample rows with only relevant columns (if they exist in the dataframe)
    lines.append(f"")
    lines.append(f"Sample rows (first 3):")
    df = table_info['df']
    # Use all columns in sample rows so model can see the data
    sample_df = df.head(3)
    lines.append(sample_df.to_string(index=False))

    return '\n'.join(lines)


print("RAG retrieval function defined!")

RAG retrieval function defined!


### RAG Prompt Template

In [59]:
def build_rag_prompt(question, table_info, table_id, col_embeddings_data, embed_model):
    """Build a prompt with RAG-retrieved context."""

    rag_context = rag_retrieve(question, table_id, table_info, col_embeddings_data, embed_model)

    prompt = f"""You are a SQL expert. Given a table and a question, generate a SQLite query to answer the question.

The following shows the table information with the most relevant columns highlighted for your question:

{rag_context}

Question: {question}

Rules:
- Return ONLY the SQL query, nothing else
- Use the exact column names provided above
- The table name is "t"
- Use SQLite syntax

SQL:"""

    return prompt


print("RAG prompt template defined!")

RAG prompt template defined!


### Test RAG Retrieval

In [60]:
# Test RAG retrieval on a few examples
print("Testing RAG retrieval...")
print("=" * 60)

for i in range(3):
    ex = eval_subset[i]
    table_info = test_tables[ex['table_id']]

    rag_context = rag_retrieve(ex['question'], ex['table_id'], table_info, col_embeddings_data, embed_model)

    print(f"Q: {ex['question']}")
    print(f"Retrieved context:")
    print(rag_context[:300])
    print("-" * 60)

Testing RAG retrieval...
Q: how many consecutive times did they play in tier 4?
Retrieved context:
Table has 12 rows and 7 columns.
Most relevant columns for your question (ranked by relevance):
  1. Level (text): example values are Tier 2, Tier 3, Tier 2 [relevance: 0.34]
  2. Movements (text): example values are Relegation Group - Relegated, Play-Offs - Promoted, Relegated [relevance: 0.24]
  3
------------------------------------------------------------
Q: how long was it between losing streak almbum and gnv fla in years.
Retrieved context:
Table has 15 rows and 5 columns.
Most relevant columns for your question (ranked by relevance):
  1. Album (text): example values are Losing Streak, Losing Streak, Non-album single [relevance: 0.33]
  2. Year (numeric): example values are 1997, 1997, 1997 [relevance: 0.21]
  3. Peak chart positions

------------------------------------------------------------
Q: who scored the least?
Retrieved context:
Table has 14 rows and 7 columns.
Most releva

### Run RAG on Eval Subset

In [61]:
# Run RAG on eval subset
if cache.exists('results/rag', 'predictions.json'):
    print("Loading RAG results from Drive cache...")
    rag_predictions = cache.load_json('results/rag', 'predictions.json')
else:
    print("Running RAG + Qwen2.5-Coder-7B on eval subset...")

    rag_predictions = []

    for i, ex in enumerate(tqdm(eval_subset, desc="RAG")):
        table_info = test_tables[ex['table_id']]
        prompt = build_rag_prompt(ex['question'], table_info, ex['table_id'], col_embeddings_data, embed_model)

        try:
            response, latency = generate_sql(prompt, model, tokenizer)
            sql = extract_sql_from_response(response)
        except Exception as e:
            sql = "SELECT 1"
            latency = 0

        rag_predictions.append({
            'id': ex['id'],
            'sql': sql,
            'latency': latency
        })

        if (i + 1) % 100 == 0:
            print(f"   Completed {i+1}/{len(eval_subset)} examples")

    cache.save_json(rag_predictions, 'results/rag', 'predictions.json')
    print("RAG predictions saved to Drive!")

Loading RAG results from Drive cache...
Loaded from Drive: results/rag/predictions.json


### Evaluate RAG

In [62]:
rag_metrics, rag_details = evaluate_predictions(
    rag_predictions, eval_subset, TEST_DB_PATH, test_tables
)

print()
print("=" * 50)
print("RAG + QWEN2.5-CODER-7B RESULTS")
print("=" * 50)
print(f"   Execution Accuracy: {rag_metrics['execution_accuracy']:.1%}")
print(f"   SQL Validity Rate:  {rag_metrics['sql_validity_rate']:.1%}")
print(f"   Correct: {rag_metrics['correct_count']} / {rag_metrics['total']}")
print(f"   Avg Latency: {rag_metrics['avg_latency']:.2f}s")
print("=" * 50)

print()
print("Full Comparison:")
print(f"   Template:          {template_metrics['execution_accuracy']:.1%}")
print(f"   Zero-shot:         {zeroshot_metrics['execution_accuracy']:.1%}")
print(f"   Few-shot:          {fewshot_metrics['execution_accuracy']:.1%}")
print(f"   CoT:               {cot_metrics['execution_accuracy']:.1%}")
print(f"   Self-Consistency:  {sc_accuracy:.1%}")
print(f"   RAG:               {rag_metrics['execution_accuracy']:.1%}")

cache.save_json(rag_metrics, 'results/rag', 'metrics.json')
cache.save_json(rag_details, 'results/rag', 'details.json')

Evaluating:   0%|          | 0/499 [00:00<?, ?it/s]


RAG + QWEN2.5-CODER-7B RESULTS
   Execution Accuracy: 43.9%
   SQL Validity Rate:  83.4%
   Correct: 219 / 499
   Avg Latency: 1.93s

Full Comparison:
   Template:          11.6%
   Zero-shot:         44.1%
   Few-shot:          45.7%
   CoT:               23.0%
   Self-Consistency:  48.1%
   RAG:               43.9%
Saved to Drive: results/rag/metrics.json
Saved to Drive: results/rag/details.json


### RAG Complexity Breakdown

In [63]:
print("RAG Results by Complexity:")
print("-" * 50)
for level in ['simple', 'medium', 'complex']:
    level_details = [d for d in rag_details if complexity_labels.get(d['id']) == level]
    if level_details:
        correct = sum(1 for d in level_details if d['correct'])
        total = len(level_details)
        print(f"   {level.capitalize():10s}: {correct}/{total} = {correct/total:.1%}")

RAG Results by Complexity:
--------------------------------------------------
   Simple    : 130/279 = 46.6%
   Medium    : 77/180 = 42.8%
   Complex   : 12/40 = 30.0%


### RAG vs all other approaches

In [64]:
# Compare RAG vs all other approaches
print("RAG vs All Approaches — Per-Example Comparison:")
print("=" * 60)

# Build lookups
zeroshot_lookup = {d['id']: d for d in zeroshot_details}
fewshot_lookup = {d['id']: d for d in fewshot_details}

comparisons = {
    'Zero-shot': zeroshot_lookup,
    'Few-shot': fewshot_lookup
}

for name, lookup in comparisons.items():
    rag_better = 0
    other_better = 0
    both_correct = 0
    both_wrong = 0

    for d in rag_details:
        other = lookup.get(d['id'])
        if other:
            if d['correct'] and other['correct']:
                both_correct += 1
            elif d['correct'] and not other['correct']:
                rag_better += 1
            elif not d['correct'] and other['correct']:
                other_better += 1
            else:
                both_wrong += 1

    print(f"\nRAG vs {name}:")
    print(f"   Both correct:    {both_correct}")
    print(f"   RAG wins:        {rag_better}")
    print(f"   {name} wins:  {other_better}")
    print(f"   Both wrong:      {both_wrong}")

# Overall summary
print()
print("=" * 60)
print("Summary:")
print(f"   Template:          {template_metrics['execution_accuracy']:.1%}")
print(f"   Zero-shot:         {zeroshot_metrics['execution_accuracy']:.1%}")
print(f"   Few-shot:          {fewshot_metrics['execution_accuracy']:.1%}")
print(f"   CoT:               {cot_metrics['execution_accuracy']:.1%}")
print(f"   Self-Consistency:  {sc_accuracy:.1%}")
print(f"   RAG:               {rag_metrics['execution_accuracy']:.1%}")

RAG vs All Approaches — Per-Example Comparison:

RAG vs Zero-shot:
   Both correct:    196
   RAG wins:        23
   Zero-shot wins:  24
   Both wrong:      256

RAG vs Few-shot:
   Both correct:    203
   RAG wins:        16
   Few-shot wins:  25
   Both wrong:      255

Summary:
   Template:          11.6%
   Zero-shot:         44.1%
   Few-shot:          45.7%
   CoT:               23.0%
   Self-Consistency:  48.1%
   RAG:               43.9%


### Section 5 Summary

In [65]:
print("=" * 60)
print("SECTION 5 COMPLETE - FULL COMPARISON SO FAR")
print("=" * 60)
print()
print(f"   {'Approach':<22} {'Accuracy':>10} {'SQL Valid':>10} {'Latency':>10}")
print(f"   {'-'*22} {'-'*10} {'-'*10} {'-'*10}")
print(f"   {'Template SQL':<22} {template_metrics['execution_accuracy']:>9.1%} {template_metrics['sql_validity_rate']:>9.1%} {template_metrics['avg_latency']*1000:>8.1f}ms")
print(f"   {'Zero-shot':<22} {zeroshot_metrics['execution_accuracy']:>9.1%} {zeroshot_metrics['sql_validity_rate']:>9.1%} {zeroshot_metrics['avg_latency']:>8.2f}s")
print(f"   {'Few-shot':<22} {fewshot_metrics['execution_accuracy']:>9.1%} {fewshot_metrics['sql_validity_rate']:>9.1%} {fewshot_metrics['avg_latency']:>8.2f}s")
print(f"   {'CoT':<22} {cot_metrics['execution_accuracy']:>9.1%} {cot_metrics['sql_validity_rate']:>9.1%} {cot_metrics['avg_latency']:>8.2f}s")
print(f"   {'Self-Consistency':<22} {sc_accuracy:>9.1%} {'—':>10} {sc_avg_latency:>8.2f}s")
print(f"   {'RAG':<22} {rag_metrics['execution_accuracy']:>9.1%} {rag_metrics['sql_validity_rate']:>9.1%} {rag_metrics['avg_latency']:>8.2f}s")
print()
print("=" * 60)
print("All results saved to Drive!")
print("Next: Section 6 - Fine-Tuning (QLoRA) — our expected best performer")
print("=" * 60)

SECTION 5 COMPLETE - FULL COMPARISON SO FAR

   Approach                 Accuracy  SQL Valid    Latency
   ---------------------- ---------- ---------- ----------
   Template SQL               11.6%     62.5%      1.4ms
   Zero-shot                  44.1%     83.0%     1.94s
   Few-shot                   45.7%     87.4%     1.89s
   CoT                        23.0%     41.7%    14.62s
   Self-Consistency           48.1%          —     6.08s
   RAG                        43.9%     83.4%     1.93s

All results saved to Drive!
Next: Section 6 - Fine-Tuning (QLoRA) — our expected best performer


## Section 6: Fine-Tuning with QLoRA

Train Qwen2.5-Coder-7B specifically on WikiTableQuestions using Parameter-Efficient Fine-Tuning. QLoRA trains only ~0.1% of parameters while keeping the base model frozen in 4-bit quantization.

**Config:** LoRA rank=8, alpha=16, target: q_proj, v_proj
**Training data:** 11,321 examples from train split
**Expected time:** 3-5 hours on A100, 8-12 hours on T4

### Prepare Training Data

In [77]:
# Prepare training data for fine-tuning
# Format: question + schema -> SQL

TRAIN_DATA_PATH = cache.cache_path('data', 'finetune_train_data.json')

if os.path.exists(TRAIN_DATA_PATH):
    print("Loading fine-tune training data from Drive cache...")
    with open(TRAIN_DATA_PATH, 'r') as f:
        finetune_data = json.load(f)
else:
    print("Preparing fine-tuning data...")
    print("Generating SQL labels using the model for training examples...")

    finetune_data = []
    skipped = 0

    for i, ex in enumerate(tqdm(train_examples[:10000], desc="Preparing training data")):
        table_info = train_tables.get(ex['table_id'])
        if table_info is None:
            skipped += 1
            continue

        schema_text = format_schema(table_info, include_samples=True, max_sample_rows=2)

        # Try to generate a correct SQL using our model
        prompt = build_zeroshot_prompt(ex['question'], table_info)
        try:
            response, _ = generate_sql(prompt, model, tokenizer)
            sql = extract_sql_from_response(response)

            # Verify the SQL produces the correct answer
            result, success, error = execute_sql(sql, ex['table_id'], TRAIN_DB_PATH)
            if success:
                match, _ = check_answer(result, ex['answers'])
                if match:
                    finetune_data.append({
                        'instruction': f"Given the following table, generate a SQL query to answer the question.\n\n{schema_text}\n\nQuestion: {ex['question']}",
                        'output': sql
                    })
        except:
            skipped += 1
            continue

        if (i + 1) % 500 == 0:
            print(f"   Processed {i+1}/5000, collected {len(finetune_data)} valid examples")

    with open(TRAIN_DATA_PATH, 'w') as f:
        json.dump(finetune_data, f)

    print(f"\nTraining data prepared!")
    print(f"   Valid examples: {len(finetune_data)}")
    print(f"   Skipped: {skipped}")

print(f"Fine-tuning dataset size: {len(finetune_data)} examples")

Loading fine-tune training data from Drive cache...
Fine-tuning dataset size: 2173 examples


### Check Training Data Sample

In [67]:
# Check a few training examples
print("Sample training examples:")
print("=" * 60)
for i in range(min(3, len(finetune_data))):
    print(f"\nExample {i+1}:")
    print(f"Instruction (first 200 chars): {finetune_data[i]['instruction'][:200]}...")
    print(f"Output (SQL): {finetune_data[i]['output']}")
    print("-" * 60)

print(f"\nTotal training examples: {len(finetune_data)}")

Sample training examples:

Example 1:
Instruction (first 200 chars): Given the following table, generate a SQL query to answer the question.

Table has 17 rows and 6 columns.
Columns:
  - year (numeric) | Examples: 2001, 2001, 2001
  - competition (text) | Examples: Wo...
Output (SQL): SELECT venue 
FROM t 
WHERE position = '1st' 
ORDER BY year DESC 
LIMIT 1;
------------------------------------------------------------

Example 2:
Instruction (first 200 chars): Given the following table, generate a SQL query to answer the question.

Table has 10 rows and 5 columns.
Columns:
  - year (numeric) | Examples: 1996, 1998, 1999
  - competition (text) | Examples: Ol...
Output (SQL): SELECT venue FROM t WHERE competition = 'Asian Games' ORDER BY year ASC LIMIT 1;
------------------------------------------------------------

Example 3:
Instruction (first 200 chars): Given the following table, generate a SQL query to answer the question.

Table has 15 rows and 6 columns.
Columns:
  - school_year 

### Setup LoRA Config

In [68]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for training
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"LoRA applied!")
print(f"   Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"   Total parameters: {total_params:,}")
print(f"   LoRA rank: {lora_config.r}")
print(f"   LoRA alpha: {lora_config.lora_alpha}")

LoRA applied!
   Trainable parameters: 2,523,136 (0.06%)
   Total parameters: 4,355,495,424
   LoRA rank: 8
   LoRA alpha: 16


### Prepare Dataset for Training

In [69]:
from torch.utils.data import Dataset as TorchDataset

class SQLFineTuneDataset(TorchDataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Format as chat template
        messages = [
            {"role": "user", "content": item['instruction']},
            {"role": "assistant", "content": item['output']}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        encodings = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        input_ids = encodings['input_ids'].squeeze()
        attention_mask = encodings['attention_mask'].squeeze()

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': input_ids.clone()
        }

# Create dataset
train_dataset = SQLFineTuneDataset(finetune_data, tokenizer, max_length=512)
print(f"Training dataset created: {len(train_dataset)} examples")

Training dataset created: 2173 examples


### Training Setup

In [70]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Training arguments
CHECKPOINT_DIR = cache.cache_path('models', 'qlora_checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Training setup complete!")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Checkpoints saved to: {CHECKPOINT_DIR}")

Training setup complete!
   Epochs: 3
   Batch size: 4
   Gradient accumulation: 4
   Effective batch size: 16
   Learning rate: 0.0002
   Checkpoints saved to: /content/drive/MyDrive/NLQuery/models/qlora_checkpoints


### Train the Model

In [71]:
# Start training (resumes from checkpoint if available)
ADAPTER_PATH = cache.cache_path('models', 'qlora_final')

if os.path.exists(ADAPTER_PATH):
    print("Fine-tuned model already exists on Drive! Skipping training.")
else:
    print("Starting/Resuming QLoRA fine-tuning...")

    # Check for existing checkpoints to resume from
    resume_checkpoint = None
    if os.path.exists(CHECKPOINT_DIR):
        checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint-')]
        if checkpoints:
            latest = sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1]
            resume_checkpoint = os.path.join(CHECKPOINT_DIR, latest)
            print(f"Resuming from checkpoint: {resume_checkpoint}")

    if resume_checkpoint:
        trainer.train(resume_from_checkpoint=resume_checkpoint)
    else:
        print("Starting fresh training...")
        trainer.train()

    # Save final adapter
    model.save_pretrained(ADAPTER_PATH)
    tokenizer.save_pretrained(ADAPTER_PATH)

    print()
    print(f"Training complete! Adapter saved to: {ADAPTER_PATH}")

Starting/Resuming QLoRA fine-tuning...
Starting fresh training...


Step,Training Loss
50,1.631075
100,0.977467
150,0.792885
200,0.727941
250,0.693570
300,0.665162
350,0.663220
400,0.661848



Training complete! Adapter saved to: /content/drive/MyDrive/NLQuery/models/qlora_final


### Load Fine-tuned Model for Inference

In [72]:
# Load the fine-tuned model (or reload if session restarted)
from peft import PeftModel

ADAPTER_PATH = cache.cache_path('models', 'qlora_final')

if not hasattr(model, 'peft_config'):
    # Need to reload base model and apply adapter
    print("Loading base model...")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-Coder-7B-Instruct",
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )

    print("Applying fine-tuned adapter...")
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

model.eval()
print("Fine-tuned model ready for inference!")

Fine-tuned model ready for inference!


### Run Fine-tuned Model on Eval Subset

In [73]:
# Run fine-tuned model on eval subset
if cache.exists('results/finetuned', 'predictions.json'):
    print("Loading fine-tuned results from Drive cache...")
    finetuned_predictions = cache.load_json('results/finetuned', 'predictions.json')
else:
    print("Running fine-tuned model on eval subset...")

    finetuned_predictions = []

    for i, ex in enumerate(tqdm(eval_subset, desc="Fine-tuned")):
        table_info = test_tables[ex['table_id']]
        prompt = build_zeroshot_prompt(ex['question'], table_info)

        try:
            response, latency = generate_sql(prompt, model, tokenizer)
            sql = extract_sql_from_response(response)
        except Exception as e:
            sql = "SELECT 1"
            latency = 0

        finetuned_predictions.append({
            'id': ex['id'],
            'sql': sql,
            'latency': latency
        })

        if (i + 1) % 100 == 0:
            print(f"   Completed {i+1}/{len(eval_subset)} examples")

    cache.save_json(finetuned_predictions, 'results/finetuned', 'predictions.json')
    print("Fine-tuned predictions saved to Drive!")

Running fine-tuned model on eval subset...


Fine-tuned:   0%|          | 0/499 [00:00<?, ?it/s]

   Completed 100/499 examples
   Completed 200/499 examples
   Completed 300/499 examples
   Completed 400/499 examples
Saved to Drive: results/finetuned/predictions.json
Fine-tuned predictions saved to Drive!


### Evaluate Fine-tuned Model

In [74]:
finetuned_metrics, finetuned_details = evaluate_predictions(
    finetuned_predictions, eval_subset, TEST_DB_PATH, test_tables
)

print()
print("=" * 60)
print("FINE-TUNED QWEN2.5-CODER-7B (QLoRA) RESULTS")
print("=" * 60)
print(f"   Execution Accuracy: {finetuned_metrics['execution_accuracy']:.1%}")
print(f"   SQL Validity Rate:  {finetuned_metrics['sql_validity_rate']:.1%}")
print(f"   Correct: {finetuned_metrics['correct_count']} / {finetuned_metrics['total']}")
print(f"   Avg Latency: {finetuned_metrics['avg_latency']:.2f}s")
print("=" * 60)

cache.save_json(finetuned_metrics, 'results/finetuned', 'metrics.json')
cache.save_json(finetuned_details, 'results/finetuned', 'details.json')

Evaluating:   0%|          | 0/499 [00:00<?, ?it/s]


FINE-TUNED QWEN2.5-CODER-7B (QLoRA) RESULTS
   Execution Accuracy: 47.3%
   SQL Validity Rate:  89.8%
   Correct: 236 / 499
   Avg Latency: 2.23s
Saved to Drive: results/finetuned/metrics.json
Saved to Drive: results/finetuned/details.json


### Fine-tuned Complexity Breakdown

In [75]:
print("Fine-tuned Results by Complexity:")
print("-" * 50)
for level in ['simple', 'medium', 'complex']:
    level_details = [d for d in finetuned_details if complexity_labels.get(d['id']) == level]
    if level_details:
        correct = sum(1 for d in level_details if d['correct'])
        total = len(level_details)
        print(f"   {level.capitalize():10s}: {correct}/{total} = {correct/total:.1%}")

Fine-tuned Results by Complexity:
--------------------------------------------------
   Simple    : 136/279 = 48.7%
   Medium    : 84/180 = 46.7%
   Complex   : 16/40 = 40.0%


### Final Full Comparison

In [76]:
print("=" * 70)
print("SECTION 6 COMPLETE - FINAL COMPARISON (ALL 7 APPROACHES)")
print("=" * 70)
print()
print(f"   {'Approach':<22} {'Accuracy':>10} {'SQL Valid':>10} {'Latency':>10}")
print(f"   {'-'*22} {'-'*10} {'-'*10} {'-'*10}")
print(f"   {'Template SQL':<22} {template_metrics['execution_accuracy']:>9.1%} {template_metrics['sql_validity_rate']:>9.1%} {template_metrics['avg_latency']*1000:>8.1f}ms")
print(f"   {'Zero-shot':<22} {zeroshot_metrics['execution_accuracy']:>9.1%} {zeroshot_metrics['sql_validity_rate']:>9.1%} {zeroshot_metrics['avg_latency']:>8.2f}s")
print(f"   {'Few-shot':<22} {fewshot_metrics['execution_accuracy']:>9.1%} {fewshot_metrics['sql_validity_rate']:>9.1%} {fewshot_metrics['avg_latency']:>8.2f}s")
print(f"   {'CoT':<22} {cot_metrics['execution_accuracy']:>9.1%} {cot_metrics['sql_validity_rate']:>9.1%} {cot_metrics['avg_latency']:>8.2f}s")
print(f"   {'Self-Consistency':<22} {sc_accuracy:>9.1%} {'--':>10} {sc_avg_latency:>8.2f}s")
print(f"   {'RAG':<22} {rag_metrics['execution_accuracy']:>9.1%} {rag_metrics['sql_validity_rate']:>9.1%} {rag_metrics['avg_latency']:>8.2f}s")
print(f"   {'Fine-tuned (QLoRA)':<22} {finetuned_metrics['execution_accuracy']:>9.1%} {finetuned_metrics['sql_validity_rate']:>9.1%} {finetuned_metrics['avg_latency']:>8.2f}s")
print()
print(f"   Improvement from baseline: {template_metrics['execution_accuracy']:.1%} -> {finetuned_metrics['execution_accuracy']:.1%}")
improvement = finetuned_metrics['execution_accuracy'] / template_metrics['execution_accuracy']
print(f"   That's {improvement:.1f}x improvement!")
print("=" * 70)
print()
print("All results saved to Drive!")
print("Next: Section 7 - Evaluation & Analysis (charts, case studies, error taxonomy)")

SECTION 6 COMPLETE - FINAL COMPARISON (ALL 7 APPROACHES)

   Approach                 Accuracy  SQL Valid    Latency
   ---------------------- ---------- ---------- ----------
   Template SQL               11.6%     62.5%      1.4ms
   Zero-shot                  44.1%     83.0%     1.94s
   Few-shot                   45.7%     87.4%     1.89s
   CoT                        23.0%     41.7%    14.62s
   Self-Consistency           48.1%         --     6.08s
   RAG                        43.9%     83.4%     1.93s
   Fine-tuned (QLoRA)         47.3%     89.8%     2.23s

   Improvement from baseline: 11.6% -> 47.3%
   That's 4.1x improvement!

All results saved to Drive!
Next: Section 7 - Evaluation & Analysis (charts, case studies, error taxonomy)
